# 00 — Setup do Ambiente no Google Colab

Este notebook prepara o ambiente de execução na GPU do Google Colab (T4 ou L4), suportando:
- **Caminho A**: notebook local conectado a um runtime do Colab via extensão da IDE.
- **Caminho B (Fallback)**: execução diretamente no Colab montando o Google Drive em `/content/drive/MyDrive/tcc2/` e sincronizando `results/` com a pasta local.


In [ ]:
# === CAMINHO B: Montar Google Drive no Colab ===
# (Pule esta célula se estiver executando localmente ou no Caminho A sem Drive)
import os
from pathlib import Path

IN_COLAB = "COLAB_GPU" in os.environ or Path("/content").exists()
USE_DRIVE = True  # Altere para False se não quiser gravar no Google Drive

DRIVE_BASE = Path("/content/drive/MyDrive/tcc2")
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    (DRIVE_BASE / "results").mkdir(parents=True, exist_ok=True)
    os.environ["TCC_RESULTS_DIR"] = str(DRIVE_BASE / "results")
    print("Resultados serão salvos em:", os.environ["TCC_RESULTS_DIR"])
else:
    print("Execução local ou sem Drive. Usando results/ do repositório.")

In [ ]:
# === Obter/instalar o pacote tcc ===
# Opção 1: clonar do Git (preencha REPO_URL se usar repositório remoto)
# Opção 2: subir o projeto compactado em /content/drive/MyDrive/tcc2/tcc2-transformers.zip
REPO_URL = "https://github.com/juniors719/codeforces-transformers.git"
REPO_DIR = Path("/content/codeforces-transformers") if IN_COLAB else Path.cwd().parent

if IN_COLAB and not REPO_DIR.exists():
    if REPO_URL:
        !git clone {REPO_URL} {REPO_DIR}
    elif (DRIVE_BASE / "tcc2-transformers.zip").exists():
        !unzip -q {DRIVE_BASE / "tcc2-transformers.zip"} -d /content/
    elif (DRIVE_BASE / "tcc2-transformers").exists():
        REPO_DIR = DRIVE_BASE / "tcc2-transformers"
    else:
        raise FileNotFoundError(
            "Informe REPO_URL ou coloque tcc2-transformers.zip (ou a pasta do projeto) em /content/drive/MyDrive/tcc2/"
        )

print("Diretório do projeto:", REPO_DIR)
%cd {REPO_DIR}

In [ ]:
# === Instalar dependências fixadas e o pacote local em modo editável ===
# Nota: no Colab, mantemos o PyTorch pré-instalado com CUDA caso já compatível.
!pip install -q transformers==4.49.0 datasets==3.3.2 accelerate==1.4.0 scikit-learn==1.6.1 scipy==1.15.2 pandas==2.2.3 matplotlib==3.10.0 PyYAML==6.0.2 tqdm==4.67.1 "sentencepiece>=0.2.0,<0.3.0" pytest==8.3.4
!pip install -q --no-deps --no-build-isolation -e .

In [ ]:
# === Verificar GPU, precisão mista (bf16/fp16) e versões ===
import importlib
import json
import sys
from pathlib import Path

src_path = str((Path.cwd() / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)
importlib.invalidate_caches()

from tcc.utils import get_environment_metadata

env_info = get_environment_metadata()
print(json.dumps(env_info, indent=2, ensure_ascii=False))

In [ ]:
# === Rodar testes unitários locais para validar instalação ===
!pytest -q